# Demo workflow: factory scheduling, disruption rescheduling, KPI comparison, and Gantt charts

Этот ноутбук продолжает предыдущие шаги и показывает полный demo-поток:

1. загрузка сгенерированных данных,
2. построение baseline-расписания,
3. пересчет расписания после остановки станка,
4. сравнение KPI,
5. визуализация результата на Gantt chart.

Ноутбук рассчитан на данные в формате, который уже был сгенерирован ранее:
- `machines.csv`
- `orders.csv`
- `operations.csv`
- `shifts.csv`
- `downtime_events.csv`
- `scenarios.csv`

И на solver-модуль:
- `cp_sat_scheduler_github_style_fixed.py`

## Что показывает сценарий

Сценарий соответствует выбранной demo-идее: **непредсказуемая остановка станка**, которая вызывает каскадный эффект и требует быстрого перепланирования. Это хорошо иллюстрирует разницу между статическим планом и real-time scheduling.

## Что понадобится

В ноутбуке используется:
- `pandas`
- `numpy`
- `matplotlib`
- `ortools`

Если `ortools` еще не установлен, следующая ячейка поставит его автоматически.


In [ ]:
# Устанавливаем только ortools, если его еще нет.
# Важно: не переустанавливаем pandas/numpy/matplotlib без необходимости,
# чтобы не сломать совместимость окружения.
import importlib.util
import sys

if importlib.util.find_spec('ortools') is None:
    !{sys.executable} -m pip install -q "ortools>=9.9,<10"
else:
    print('ortools is already installed')


## 1. Импорт библиотек и настройка путей

По умолчанию ноутбук смотрит на synthetic-demo bundle, который уже был создан ранее.  
При желании можно переключиться на `embedded_benchmark`.


In [ ]:
from pathlib import Path
import sys
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Путь до папки с данными
DATA_ROOT = Path("generated_factory_demo_data")
BUNDLE_NAME = "synthetic_demo"   # можно заменить на "embedded_benchmark"
BUNDLE_DIR = DATA_ROOT / BUNDLE_NAME

# Путь до solver-модуля
MODULE_PATH = Path("cp_sat_scheduler_github_style_fixed.py")

print("Bundle dir:", BUNDLE_DIR.resolve())
print("Module path:", MODULE_PATH.resolve())

if not MODULE_PATH.exists():
    raise FileNotFoundError(
        "Файл cp_sat_scheduler_github_style_fixed.py не найден рядом с ноутбуком. "
        "Скачай его вместе с ноутбуком или поправь путь MODULE_PATH."
    )

sys.path.append(str(MODULE_PATH.parent.resolve()))

from cp_sat_scheduler_github_style_fixed import (
    load_data_bundle,
    solve_schedule,
    run_reschedule_on_event,
    compute_kpis,
    export_schedule_csv,
    validate_schedule,
)


## 2. Загрузка данных и быстрый обзор структуры

Сначала загрузим bundle и посмотрим, как выглядят таблицы. Это полезно, чтобы убедиться, что структура соответствует ожиданиям solver-а.


In [ ]:
bundle = load_data_bundle(BUNDLE_DIR)

print("Machines:", bundle.machines.shape)
print("Orders:", bundle.orders.shape)
print("Operations:", bundle.operations.shape)
print("Shifts:", bundle.shifts.shape)
print("Downtime events:", bundle.downtime_events.shape)
print("Scenarios:", bundle.scenarios.shape)

print("\nMachines preview:")
print(bundle.machines.head().to_string(index=False))
print("\nOrders preview:")
print(bundle.orders.head().to_string(index=False))
print("\nOperations preview:")
print(bundle.operations.head().to_string(index=False))
print("\nScenarios:")
print(bundle.scenarios.to_string(index=False))


## 3. Решение baseline-задачи

Здесь мы строим исходное расписание без disruption.  
Это и будет наш **reference plan**, с которым потом сравнивается repaired schedule.

> Замечание: solver использует CP-SAT и учитывает:
> - no-overlap по станкам,
> - последовательность операций внутри заказа,
> - дедлайны и приоритеты,
> - смены,
> - назначение на допустимые группы оборудования.


In [ ]:
baseline = solve_schedule(
    BUNDLE_DIR,
    scenario_name="baseline_no_disruption",
    time_limit_seconds=20,
    num_search_workers=8,
)

print("Baseline status:", baseline.status)
print("Baseline objective:", baseline.objective_value)
print("Baseline solve time [s]:", round(baseline.solve_time_seconds, 3))

if baseline.status != "OPTIMAL":
    print("Warning: baseline solve finished as FEASIBLE, not OPTIMAL.")
    print("This schedule is valid, but it is not proven globally optimal.")

if baseline.status not in {"OPTIMAL", "FEASIBLE"}:
    raise RuntimeError("Baseline schedule was not solved successfully.")

print("\nBaseline schedule head:")
print(baseline.schedule.head(20).to_string(index=False))
print("\nBaseline order summary head:")
print(baseline.order_summary.head(20).to_string(index=False))
print("\nNaN machine_id in baseline:", baseline.schedule["machine_id"].isna().sum())

baseline_validation = validate_schedule(
    baseline.schedule,
    bundle,
    scenario_name='baseline_no_disruption',
)
print("\nBaseline validation:")
print(pd.DataFrame([baseline_validation]).T.rename(columns={0: 'value'}).to_string())


## 4. Вычисление KPI для baseline

Считаем базовые метрики:
- makespan,
- total tardiness,
- число просроченных заказов,
- priority-weighted tardiness,
- machine idle time.


In [ ]:
baseline_kpis = compute_kpis(
    baseline.schedule,
    bundle.orders,
    bundle.operations,
    bundle.shifts,
)

baseline_kpis_df = pd.DataFrame([baseline_kpis]).T.reset_index()
baseline_kpis_df.columns = ["metric", "baseline_value"]
print(baseline_kpis_df.to_string(index=False))


## 5. Просмотр доступных disruption-сценариев

В bundle уже лежит несколько сценариев:
- `baseline_no_disruption`
- `optimistic_estimate`
- `pessimistic_estimate`
- `updated_after_10_min`

Обычно для demo удобно начать с `optimistic_estimate`, а затем показать, как результат меняется при более плохой оценке или после уточнения информации.


In [ ]:
print(bundle.scenarios.to_string(index=False))

SCENARIO_NAME = "optimistic_estimate"
USE_ACTUAL_DOWNTIME = False   # True — использовать actual_duration_minutes, False — estimated_duration_minutes

print("Selected scenario:", SCENARIO_NAME)
print("Use actual downtime:", USE_ACTUAL_DOWNTIME)

print(bundle.downtime_events[bundle.downtime_events["scenario_name"] == SCENARIO_NAME].to_string(index=False))


## 6. Перепланирование после события

Теперь используем уже построенный baseline-план и запускаем **rolling rescheduling** после события остановки станка.

Важная настройка:
- `freeze_started_operations=True` означает, что уже начатые операции не прерываются и сохраняются как есть.  
Это соответствует типичному non-preemptive производственному сценарию.


In [ ]:
repaired = run_reschedule_on_event(
    BUNDLE_DIR,
    baseline.schedule,
    scenario_name=SCENARIO_NAME,
    freeze_started_operations=True,
    use_actual_downtime=USE_ACTUAL_DOWNTIME,
    time_limit_seconds=20,
    num_search_workers=8,
)

print("Repair status:", repaired.status)
print("Repair objective:", repaired.objective_value)
print("Repair solve time [s]:", round(repaired.solve_time_seconds, 3))

if repaired.status != "OPTIMAL":
    print("Warning: repaired solve finished as FEASIBLE, not OPTIMAL.")
    print("This means KPI/objective comparison is useful for demo purposes,")
    print("but should not be interpreted as a proof that repaired is globally better.")

if repaired.status not in {"OPTIMAL", "FEASIBLE"}:
    raise RuntimeError("Repaired schedule was not solved successfully.")

print("\nRepaired schedule head:")
print(repaired.schedule.head(20).to_string(index=False))
print("\nRepaired order summary head:")
print(repaired.order_summary.head(20).to_string(index=False))
print("\nNaN machine_id in repaired:", repaired.schedule["machine_id"].isna().sum())

repaired_validation = validate_schedule(
    repaired.schedule,
    bundle,
    scenario_name=SCENARIO_NAME,
    use_actual_downtime=USE_ACTUAL_DOWNTIME,
    replan_time=repaired.metadata.get('replan_time'),
)
print("\nRepaired validation:")
print(pd.DataFrame([repaired_validation]).T.rename(columns={0: 'value'}).to_string())


## 7. KPI после перепланирования и сравнение с baseline

Теперь считаем метрики для repaired schedule и отдельно stability-метрики относительно baseline:
- средний сдвиг операций,
- число изменившихся операций.


In [ ]:
repaired_kpis = compute_kpis(
    repaired.schedule,
    bundle.orders,
    bundle.operations,
    bundle.shifts,
    previous_schedule_df=baseline.schedule,
)

comparison_df = pd.DataFrame({
    "metric": sorted(set(baseline_kpis.keys()) | set(repaired_kpis.keys()))
})
comparison_df["baseline"] = comparison_df["metric"].map(baseline_kpis)
comparison_df["repaired"] = comparison_df["metric"].map(repaired_kpis)
comparison_df["delta_repaired_minus_baseline"] = comparison_df["repaired"] - comparison_df["baseline"]

print(comparison_df.to_string(index=False))


## 8. Экспорт расписаний

Если нужно использовать результат дальше в UI или backend, можно сохранить baseline и repaired schedules в CSV.


In [ ]:
OUTPUT_DIR = Path("scheduler_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

baseline_path = OUTPUT_DIR / f"{BUNDLE_NAME}_baseline_schedule.csv"
repaired_path = OUTPUT_DIR / f"{BUNDLE_NAME}_{SCENARIO_NAME}_repaired_schedule.csv"

export_schedule_csv(baseline.schedule, baseline_path)
export_schedule_csv(repaired.schedule, repaired_path)

print("Saved:", baseline_path.resolve())
print("Saved:", repaired_path.resolve())


## 9. Вспомогательная функция для Gantt chart

Ниже простая функция визуализации:
- каждая строка — отдельный станок,
- цвет — заказ,
- слева baseline или repaired,
- downtime можно нанести отдельной полупрозрачной заливкой.


In [ ]:
def plot_gantt_v2(
    schedule_df,
    title,
    downtime_df=None,
    shifts_df=None,
    event_time=None,
    label_mode="order",   # operation | order | none
    min_label_minutes=85,
    machine_order=None,
    downtime_duration_col="estimated_duration_minutes",
    figsize=(18, 7),
    xlim_mode="shift_span",   # shift_span | data_span
    edge_padding_minutes=15,
):
    df = schedule_df.copy()
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["end_time"] = pd.to_datetime(df["end_time"])

    if machine_order is None:
        machines = list(df["machine_id"].dropna().astype(str).sort_values().unique())
    else:
        machines = list(machine_order)

    machine_to_y = {m: i for i, m in enumerate(machines)}

    fig, ax = plt.subplots(figsize=figsize)

    order_ids = sorted(df["order_id"].astype(str).unique())
    cmap = plt.colormaps["tab20"]
    order_to_color = {oid: cmap(i % 20) for i, oid in enumerate(order_ids)}

    # ---------- Основные прямоугольники операций ----------
    for _, row in df.iterrows():
        machine_id = row["machine_id"]
        if pd.isna(machine_id) or str(machine_id) not in machine_to_y:
            continue

        y = machine_to_y[str(machine_id)]
        start = mdates.date2num(row["start_time"])
        end = mdates.date2num(row["end_time"])
        width = end - start
        duration_min = (row["end_time"] - row["start_time"]).total_seconds() / 60.0

        ax.barh(
            y=y,
            width=width,
            left=start,
            height=0.62,
            color=order_to_color[str(row["order_id"])],
            edgecolor="black",
            linewidth=0.8,
            alpha=0.9,
        )

        if duration_min >= min_label_minutes and label_mode != "none":
            if label_mode == "operation":
                label = f"{row['order_id']} / op{row['sequence_index']}"
            else:
                label = str(row["order_id"])

            ax.text(
                start + width / 2,
                y,
                label,
                ha="center",
                va="center",
                fontsize=8,
                color="black",
            )

    # ---------- Downtime-окна ----------
    downtime_legend_needed = False
    event_times = []

    if downtime_df is not None and not downtime_df.empty:
        tmp = downtime_df.copy()
        tmp["event_start"] = pd.to_datetime(tmp["event_start"])
        event_times.extend(tmp["event_start"].dropna().tolist())

        for _, row in tmp.iterrows():
            machine_id = str(row["machine_id"])
            if machine_id not in machine_to_y:
                continue

            duration = row.get(downtime_duration_col, None)
            if pd.isna(duration):
                continue

            y = machine_to_y[machine_id]
            start = mdates.date2num(row["event_start"])
            end = mdates.date2num(
                row["event_start"] + pd.Timedelta(minutes=float(duration))
            )

            ax.barh(
                y=y,
                width=end - start,
                left=start,
                height=0.84,
                color="red",
                alpha=0.18,
                edgecolor="red",
                hatch="//",
                linewidth=1.0,
            )
            downtime_legend_needed = True

    # Явно переданный момент события тоже рисуем вертикальной линией.
    if event_time is not None and not pd.isna(event_time):
        event_times.append(pd.to_datetime(event_time))

    unique_event_times = sorted(pd.to_datetime(pd.Series(event_times).dropna().unique())) if event_times else []

    event_line_needed = False
    for idx, ts in enumerate(unique_event_times):
        x = mdates.date2num(pd.to_datetime(ts))
        ax.axvline(
            x=x,
            linestyle="--",
            linewidth=1.6,
            alpha=0.9,
        )
        label = "Downtime event" if idx == 0 else f"Event {idx + 1}"
        ax.text(
            x,
            len(machines) - 0.35 if machines else 0.0,
            label,
            rotation=90,
            ha="right",
            va="top",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7, edgecolor="none"),
        )
        event_line_needed = True

    # ---------- Границы по оси X ----------
    display_start = None
    display_end = None

    if xlim_mode == "shift_span" and shifts_df is not None and not shifts_df.empty:
        tmp_shifts = shifts_df.copy()
        tmp_shifts["shift_start"] = pd.to_datetime(tmp_shifts["shift_start"])
        tmp_shifts["shift_end"] = pd.to_datetime(tmp_shifts["shift_end"])

        # Берём все рабочие смены по станкам, которые реально отображаются на графике.
        if "is_working" in tmp_shifts.columns:
            working_mask = tmp_shifts["is_working"].fillna(False).astype(bool)
            tmp_shifts = tmp_shifts[working_mask]

        tmp_shifts = tmp_shifts[tmp_shifts["machine_id"].astype(str).isin(machines)]

        if not tmp_shifts.empty:
            display_start = tmp_shifts["shift_start"].min()
            display_end = tmp_shifts["shift_end"].max()

    if display_start is None or display_end is None:
        candidates_start = [df["start_time"].min()]
        candidates_end = [df["end_time"].max()]

        if downtime_df is not None and not downtime_df.empty:
            tmp = downtime_df.copy()
            tmp["event_start"] = pd.to_datetime(tmp["event_start"])
            candidates_start.append(tmp["event_start"].min())

            if downtime_duration_col in tmp.columns:
                dt_end = tmp["event_start"] + pd.to_timedelta(tmp[downtime_duration_col], unit="m")
                candidates_end.append(dt_end.max())

        if unique_event_times:
            candidates_start.append(min(unique_event_times))
            candidates_end.append(max(unique_event_times))

        display_start = min(pd.to_datetime(pd.Series(candidates_start).dropna()))
        display_end = max(pd.to_datetime(pd.Series(candidates_end).dropna()))

    display_start = pd.to_datetime(display_start) - pd.Timedelta(minutes=edge_padding_minutes)
    display_end = pd.to_datetime(display_end) + pd.Timedelta(minutes=edge_padding_minutes)

    ax.set_xlim(mdates.date2num(display_start), mdates.date2num(display_end))

    # ---------- Оформление ----------
    ax.set_title(title, fontsize=14)
    ax.set_yticks(range(len(machines)))
    ax.set_yticklabels(machines, fontsize=10)
    ax.set_ylabel("Machine", fontsize=11)
    ax.set_xlabel("Time", fontsize=11)
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    plt.xticks(rotation=30, fontsize=10)

    legend_handles = []
    if downtime_legend_needed:
        legend_handles.append(
            Patch(
                facecolor="red",
                edgecolor="red",
                alpha=0.18,
                hatch="//",
                label="Downtime window",
            )
        )
    if event_line_needed:
        legend_handles.append(
            Line2D(
                [0],
                [0],
                linestyle="--",
                linewidth=1.6,
                label="Downtime event time",
            )
        )

    if legend_handles:
        ax.legend(handles=legend_handles, loc="upper right")

    plt.tight_layout()
    plt.show()

## 10. Визуализация baseline-расписания

Это исходный план до нарушения.


In [ ]:
machine_order = sorted(
    set(baseline.schedule["machine_id"].dropna().astype(str).unique()).union(
        set(repaired.schedule["machine_id"].dropna().astype(str).unique())
    )
)

scenario_downtime = bundle.downtime_events[
    bundle.downtime_events["scenario_name"] == SCENARIO_NAME
].copy()
event_time_for_plot = None
if not scenario_downtime.empty:
    event_time_for_plot = pd.to_datetime(scenario_downtime["event_start"]).min()

plot_gantt_v2(
    baseline.schedule,
    title=f"Baseline schedule — {BUNDLE_NAME}",
    machine_order=machine_order,
    label_mode="order",
    min_label_minutes=85,
    downtime_df=None,
    shifts_df=bundle.shifts,
    event_time=event_time_for_plot,
    xlim_mode="shift_span",
    edge_padding_minutes=15,
)

## 11. Визуализация repaired-расписания с downtime

Здесь уже виден эффект disruption и последующего перепланирования.


In [ ]:
scenario_downtime = bundle.downtime_events[
    bundle.downtime_events["scenario_name"] == SCENARIO_NAME
].copy()

event_time_for_plot = None
if not scenario_downtime.empty:
    event_time_for_plot = pd.to_datetime(scenario_downtime["event_start"]).min()

plot_gantt_v2(
    repaired.schedule,
    title=f"Repaired schedule — {BUNDLE_NAME} — {SCENARIO_NAME}",
    machine_order=machine_order,
    label_mode="order",
    min_label_minutes=85,
    downtime_df=scenario_downtime,
    shifts_df=bundle.shifts,
    event_time=event_time_for_plot,
    downtime_duration_col=("actual_duration_minutes" if USE_ACTUAL_DOWNTIME else "estimated_duration_minutes"),
    xlim_mode="shift_span",
    edge_padding_minutes=15,
)

## 12. Визуальное сравнение baseline и repaired только для изменившихся операций

Чтобы показать ripple effect более наглядно, удобно посмотреть, какие операции реально сдвинулись.


In [ ]:
compare = repaired.schedule.merge(
    baseline.schedule[["operation_id", "machine_id", "start_time", "end_time"]].rename(columns={
        "machine_id": "baseline_machine_id",
        "start_time": "baseline_start_time",
        "end_time": "baseline_end_time",
    }),
    on="operation_id",
    how="left",
)

compare["baseline_start_time"] = pd.to_datetime(compare["baseline_start_time"])
compare["baseline_end_time"] = pd.to_datetime(compare["baseline_end_time"])
compare["start_time"] = pd.to_datetime(compare["start_time"])
compare["end_time"] = pd.to_datetime(compare["end_time"])

compare["start_shift_minutes"] = (
    (compare["start_time"] - compare["baseline_start_time"]).dt.total_seconds().abs() / 60.0
)
compare["machine_changed"] = compare["machine_id"] != compare["baseline_machine_id"]
compare["changed"] = compare["machine_changed"] | (compare["start_shift_minutes"] > 0.0)

changed_ops = compare[compare["changed"]].sort_values(
    "start_shift_minutes", ascending=False
)

print(
    changed_ops[
        [
            "operation_id",
            "order_id",
            "baseline_machine_id",
            "machine_id",
            "baseline_start_time",
            "start_time",
            "start_shift_minutes",
            "machine_changed",
        ]
    ].head(30).to_string(index=False)
)

print("Changed operations:", len(changed_ops))
print("Average operation shift [min]:", changed_ops["start_shift_minutes"].mean())


## 13. Маленькая business-интерпретация результата

Ниже ячейка собирает короткий текстовый summary, который уже можно использовать почти как пояснение к demo-слайду.


In [ ]:
def summarize_demo_result(
    baseline_kpis,
    repaired_kpis,
    scenario_name,
    baseline_status=None,
    repaired_status=None,
):
    lines = []
    lines.append(f"Scenario: {scenario_name}")
    lines.append(
        f"Makespan changed from {baseline_kpis['makespan_minutes']:.1f} "
        f"to {repaired_kpis['makespan_minutes']:.1f} minutes."
    )
    lines.append(
        f"Total tardiness changed from {baseline_kpis['total_tardiness_minutes']:.1f} "
        f"to {repaired_kpis['total_tardiness_minutes']:.1f} minutes."
    )
    lines.append(
        f"Late orders changed from {baseline_kpis['late_orders']:.0f} "
        f"to {repaired_kpis['late_orders']:.0f}."
    )

    if not np.isnan(repaired_kpis.get("changed_operations_vs_previous", np.nan)):
        lines.append(
            f"Ripple effect: {repaired_kpis['changed_operations_vs_previous']:.0f} "
            f"operations changed relative to baseline."
        )

    if not np.isnan(repaired_kpis.get("average_operation_shift_minutes_vs_previous", np.nan)):
        lines.append(
            f"Average operation shift relative to baseline: "
            f"{repaired_kpis['average_operation_shift_minutes_vs_previous']:.1f} minutes."
        )

    if baseline_status != "OPTIMAL" or repaired_status != "OPTIMAL":
        lines.append(
            "Note: at least one schedule is FEASIBLE rather than OPTIMAL, "
            "so this comparison is useful for demo purposes but is not a proof of global superiority."
        )

    return "\n".join(lines)

print(
    summarize_demo_result(
        baseline_kpis,
        repaired_kpis,
        SCENARIO_NAME,
        baseline_status=baseline.status,
        repaired_status=repaired.status,
    )
)


## 14. Идеи для следующего шага

После этого ноутбука уже легко перейти к:
- сравнению нескольких disruption-сценариев в цикле,
- сравнению classical vs heuristic baseline,
- выгрузке JSON/CSV в frontend,
- добавлению quantum / quantum-inspired solver для side-by-side comparison.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=94ee51bb-3d3f-496f-b8a5-70cdfc1e8f56' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>